# Objective perplexity evals for sequence generation

This notebook rewrites the sequence-generation example around an objective, reproducible evaluation: **perplexity**.

Perplexity is the exponentiated average negative log-likelihood of a reference sequence under a model:

$$\text{perplexity}=\exp\left(-\frac{1}{N}\sum_{i=1}^N \log p(x_i \mid x_{<i})\right).$$

Lower perplexity means the model assigns higher probability to the observed continuation. In this toy notebook we can inspect every probability, so perplexity is a transparent stand-in for objective model-based evaluations used with real language models.

We will:

1. Build a tiny sequence dataset from a known probabilistic grammar.
2. Fit several simple sequence models.
3. Evaluate them with held-out negative log-likelihood and perplexity.
4. Compare generated samples using the same objective evaluator.
5. Show why validation/test perplexity is more trustworthy than anecdotes from a few generations.

In [1]:
import math
import random
from collections import Counter

random.seed(11)

ALPHABET = list("ABCD")
START = "<s>"

# A small ground-truth Markov grammar.  A and C usually alternate with B/D,
# but there is enough noise that all tokens remain possible.
TRUE_TRANSITIONS = {
    START: {"A": 0.62, "B": 0.18, "C": 0.12, "D": 0.08},
    "A": {"B": 0.64, "C": 0.22, "D": 0.10, "A": 0.04},
    "B": {"A": 0.46, "C": 0.34, "D": 0.15, "B": 0.05},
    "C": {"D": 0.58, "A": 0.25, "B": 0.12, "C": 0.05},
    "D": {"A": 0.53, "C": 0.24, "B": 0.16, "D": 0.07},
}

def draw_from(dist):
    r = random.random()
    total = 0.0
    for token, prob in dist.items():
        total += prob
        if r <= total:
            return token
    return token

def sample_true_sequence(length=18):
    prev = START
    out = []
    for _ in range(length):
        token = draw_from(TRUE_TRANSITIONS[prev])
        out.append(token)
        prev = token
    return "".join(out)

sequences = [sample_true_sequence() for _ in range(900)]
train = sequences[:600]
validation = sequences[600:750]
test = sequences[750:]

print(f"Train/validation/test sizes: {len(train)}/{len(validation)}/{len(test)}")
print("First five training sequences:")
for seq in train[:5]:
    print(" ", seq)

Train/validation/test sizes: 600/150/150
First five training sequences:
  ABDABCDABCDABDCDDD
  BCDABABABCDBCABCDA
  DDBCDABACDBADBABDA
  DABCABABBCDABACDCD
  ACDCDABAACDBABDCDD


## Perplexity helper

The functions below compute token-level log-likelihood, average negative log-likelihood (NLL), and perplexity. We use add-α smoothing so every candidate model assigns non-zero probability to every possible next token.

In [2]:
def sequence_log_probability(sequence, conditional_probability):
    prev = START
    logp = 0.0
    for token in sequence:
        p = conditional_probability(prev, token)
        logp += math.log(p)
        prev = token
    return logp

def corpus_metrics(corpus, conditional_probability):
    token_count = sum(len(seq) for seq in corpus)
    total_logp = sum(sequence_log_probability(seq, conditional_probability) for seq in corpus)
    nll = -total_logp / token_count
    return {"tokens": token_count, "nll": nll, "perplexity": math.exp(nll)}

def print_metrics(name, model):
    rows = []
    for split_name, split in [("train", train), ("validation", validation), ("test", test)]:
        m = corpus_metrics(split, model)
        rows.append((split_name, m["nll"], m["perplexity"]))
    print(name)
    print("split       nll/token   perplexity")
    for split_name, nll, ppl in rows:
        print(f"{split_name:10s} {nll:9.3f} {ppl:12.3f}")
    print()

## Candidate sequence models

We compare four models:

- **Uniform**: every token is equally likely.
- **Unigram**: learns overall token frequencies but ignores context.
- **Bigram**: learns next-token probabilities conditioned on the previous token.
- **Overfit bigram**: uses almost no smoothing; it can look great on training data but is less robust on held-out data.

The objective evaluation will decide which model predicts unseen sequences best.

In [3]:
def fit_unigram(corpus, alpha=0.5):
    counts = Counter(token for seq in corpus for token in seq)
    denominator = sum(counts.values()) + alpha * len(ALPHABET)
    probs = {token: (counts[token] + alpha) / denominator for token in ALPHABET}
    return lambda prev, token: probs[token]

def fit_bigram(corpus, alpha=0.5):
    counts = {prev: Counter() for prev in [START] + ALPHABET}
    totals = Counter()
    for seq in corpus:
        prev = START
        for token in seq:
            counts[prev][token] += 1
            totals[prev] += 1
            prev = token
    def prob(prev, token):
        return (counts[prev][token] + alpha) / (totals[prev] + alpha * len(ALPHABET))
    return prob

uniform_model = lambda prev, token: 1 / len(ALPHABET)
unigram_model = fit_unigram(train, alpha=0.5)
bigram_model = fit_bigram(train, alpha=0.5)
overfit_bigram_model = fit_bigram(train[:60], alpha=1e-6)

for name, model in [
    ("Uniform model", uniform_model),
    ("Unigram model", unigram_model),
    ("Bigram model", bigram_model),
    ("Overfit bigram model", overfit_bigram_model),
]:
    print_metrics(name, model)

Uniform model
split       nll/token   perplexity
train          1.386        4.000
validation     1.386        4.000
test           1.386        4.000

Unigram model
split       nll/token   perplexity
train          1.369        3.930
validation     1.370        3.934
test           1.367        3.925

Bigram model
split       nll/token   perplexity
train          1.075        2.930
validation     1.082        2.950
test           1.072        2.921

Overfit bigram model
split       nll/token   perplexity
train          1.085        2.961
validation     1.093        2.982
test           1.086        2.964



## Inspecting probabilities keeps the evaluation honest

Perplexity is not a subjective preference score. It is computed from the probabilities a model assigns to actual held-out tokens. Here are the learned bigram probabilities next to the true data-generating probabilities.

In [4]:
def learned_row(model, prev):
    return {token: round(model(prev, token), 3) for token in ALPHABET}

print("prev | true probabilities                    | learned bigram probabilities")
for prev in [START] + ALPHABET:
    true_row = {token: round(TRUE_TRANSITIONS[prev][token], 3) for token in ALPHABET}
    print(f"{prev:>3s} | {true_row} | {learned_row(bigram_model, prev)}")

prev | true probabilities                    | learned bigram probabilities
<s> | {'A': 0.62, 'B': 0.18, 'C': 0.12, 'D': 0.08} | {'A': 0.624, 'B': 0.162, 'C': 0.134, 'D': 0.081}
  A | {'A': 0.04, 'B': 0.64, 'C': 0.22, 'D': 0.1} | {'A': 0.042, 'B': 0.647, 'C': 0.214, 'D': 0.097}
  B | {'A': 0.46, 'B': 0.05, 'C': 0.34, 'D': 0.15} | {'A': 0.484, 'B': 0.049, 'C': 0.326, 'D': 0.142}
  C | {'A': 0.25, 'B': 0.12, 'C': 0.05, 'D': 0.58} | {'A': 0.244, 'B': 0.119, 'C': 0.048, 'D': 0.589}
  D | {'A': 0.53, 'B': 0.16, 'C': 0.24, 'D': 0.07} | {'A': 0.517, 'B': 0.166, 'C': 0.238, 'D': 0.079}


## Evaluating generated samples

Perplexity normally evaluates a model on reference data. We can also score generated samples under a fixed evaluator. Below, the fitted bigram model acts as an objective evaluator: samples that resemble the learned grammar get lower perplexity, while samples that violate it get higher perplexity.

In [5]:
def generate(model, length=18, greedy=False):
    prev = START
    out = []
    for _ in range(length):
        probs = {token: model(prev, token) for token in ALPHABET}
        if greedy:
            token = max(probs, key=probs.get)
        else:
            token = draw_from(probs)
        out.append(token)
        prev = token
    return "".join(out)

def eval_sequences(label, generated, evaluator=bigram_model):
    m = corpus_metrics(generated, evaluator)
    print(f"{label:24s} perplexity={m['perplexity']:.3f}  nll/token={m['nll']:.3f}")
    for seq in generated[:3]:
        print("  ", seq)
    print()

random_samples = [generate(uniform_model) for _ in range(30)]
unigram_samples = [generate(unigram_model) for _ in range(30)]
bigram_samples = [generate(bigram_model) for _ in range(30)]
greedy_bigram_samples = [generate(bigram_model, greedy=True) for _ in range(30)]

eval_sequences("Uniform samples", random_samples)
eval_sequences("Unigram samples", unigram_samples)
eval_sequences("Bigram samples", bigram_samples)
eval_sequences("Greedy bigram samples", greedy_bigram_samples)

Uniform samples          perplexity=5.585  nll/token=1.720
   DAADDBCDBBCACDCACA
   ACACBCDAAABDACBBCC
   DDDADBBABBDBAACCCB

Unigram samples          perplexity=6.038  nll/token=1.798
   CCCBBDADACDCCBBDAD
   CACAACAABDBCAAAAAA
   DAAABCBCACBADBDDDA

Bigram samples           perplexity=2.894  nll/token=1.063
   ACDCCDABABCDACABAD
   CDABADABBABDCBCDCB
   CABABABABCDCACABCA

Greedy bigram samples    perplexity=1.762  nll/token=0.566
   ABABABABABABABABAB
   ABABABABABABABABAB
   ABABABABABABABABAB



## Takeaways

- Perplexity is an objective, repeatable sequence-level metric derived from likelihood.
- Lower perplexity means a model assigns more probability to the evaluated tokens.
- Held-out validation/test perplexity is the useful number; training perplexity alone can reward memorization or under-smoothed models.
- Sampling anecdotes are helpful for debugging, but perplexity makes model comparisons auditable.
- Perplexity is still not the same as human preference or task success, so production LLM evaluation often combines it with exact-match tests, preference judgments, safety checks, and domain-specific verifiers.